In [86]:
import pandas as pd
import sqlite3

In [87]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

In [88]:
schema_query = "PRAGMA table_info(test);"
schema = pd.io.sql.read_sql(schema_query, conn)
print("Схема таблицы test:")
print(schema)

Схема таблицы test:
   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


In [89]:
first_10_query = "SELECT * FROM test LIMIT 10;"
first_10_rows = pd.io.sql.read_sql(first_10_query, conn)
print("\nПервые 10 строк таблицы test:")
print(first_10_rows)


Первые 10 строк таблицы test:
       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


In [90]:
deadlines_schema = "PRAGMA table_info(deadlines);"
deadlines_info = pd.io.sql.read_sql(deadlines_schema, conn)
print("\nСхема таблицы deadlines:")
print(deadlines_info)


Схема таблицы deadlines:
   cid       name     type  notnull dflt_value  pk
0    0      index  INTEGER        0       None   0
1    1       labs     TEXT        0       None   0
2    2  deadlines  INTEGER        0       None   0


In [91]:
deadlines_data = pd.io.sql.read_sql("SELECT * FROM deadlines;", conn)
print("\nДанные таблицы deadlines:")
print(deadlines_data)


Данные таблицы deadlines:
   index      labs   deadlines
0      0    laba04  1587945599
1      1   laba04s  1587945599
2      2    laba05  1588550399
3      4    laba06  1590364799
4      5   laba06s  1590364799
5      3  project1  1589673599


## Минимальная дельта

In [92]:
min_query = """
SELECT 
    t.uid,
    MIN(CAST((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0 AS INTEGER)) as delta_hours
FROM test t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
    AND t.first_commit_ts IS NOT NULL
GROUP BY t.uid
ORDER BY delta_hours ASC
LIMIT 1;
"""

df_min = pd.io.sql.read_sql(min_query, conn)
print("Минимальная дельта (в часах):")
print(df_min)

Минимальная дельта (в часах):
       uid  delta_hours
0  user_30         -202


## Максимальная дельта

In [93]:
max_query = """
SELECT 
    t.uid,
    MAX(CAST((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0 AS INTEGER)) as delta_hours
FROM test t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
    AND t.first_commit_ts IS NOT NULL
GROUP BY t.uid
ORDER BY delta_hours DESC
LIMIT 1;
"""

df_max = pd.io.sql.read_sql(max_query, conn)
print("\nМаксимальная дельта (в часах):")
print(df_max)


Максимальная дельта (в часах):
       uid  delta_hours
0  user_25           -2


## Средняя дельта

In [94]:
avg_query = """
SELECT 
    AVG(CAST((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0 AS REAL)) as avg_delta_hours
FROM test t
JOIN deadlines d ON t.labname = d.labs
WHERE t.labname != 'project1'
    AND t.first_commit_ts IS NOT NULL;
"""

df_avg = pd.io.sql.read_sql(avg_query, conn)
print("Средняя дельта (в часах):")
print(df_avg)

Средняя дельта (в часах):
   avg_delta_hours
0       -89.687841


## Анализ корреляции между числом просмотров и дельтой

In [95]:
views_diff_query = """
SELECT 
    t.uid,
    AVG(CAST((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0 AS REAL)) as avg_diff,
    COUNT(p.datetime) as pageviews
FROM test t
JOIN deadlines d ON t.labname = d.labs
LEFT JOIN pageviews p ON t.uid = p.uid
WHERE t.labname != 'project1'
    AND t.first_commit_ts IS NOT NULL
GROUP BY t.uid;
"""

views_diff = pd.io.sql.read_sql(views_diff_query, conn)
print("Таблица views_diff (первые 10 строк):")
print(views_diff.head(10))


Таблица views_diff (первые 10 строк):
       uid    avg_diff  pageviews
0   user_1  -65.119778        140
1  user_10  -75.242444        445
2  user_14 -159.568796        429
3  user_17  -62.207667        235
4  user_18   -6.368148          9
5  user_19  -99.440417         64
6  user_21  -96.111181         40
7  user_25  -93.474944        895
8  user_28  -86.793833        745
9   user_3 -105.738222       1585


In [96]:
views_diff[['avg_diff', 'pageviews']].corr()


,avg_diff,pageviews
avg_diff,1.000000,-0.185042
pageviews,-0.185042,1.000000


In [97]:

# Проверяем информацию о таблице
print("\nИнформация о views_diff:")
print(views_diff.info())



Информация о views_diff:
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   uid        11 non-null     str    
 1   avg_diff   11 non-null     float64
 2   pageviews  11 non-null     int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 396.0 bytes
None


In [98]:
conn.close()